## Data Loading Functions

This section defines functions to load phylogenetic graph data from CSV files and convert them to DGL format.

**Functions:**
- `parse_edge_feature()`: Parses edge feature strings (format: `[val1\tval2\t...]`) into float vectors
- `load_and_convert_to_dgl()`: Main loading function that:
  1. Loads all `*_node.csv` and `*_edge.csv` files from input folder
  2. Filters for fully connected graphs with specified number of nodes
  3. Filters graphs by edge feature vector length constraints
  4. Pads edge features to uniform length using max value
  5. Converts to DGL bidirectional graphs with node/edge features
- `main()`: Entry point that orchestrates the loading pipeline and prints summary statistics

**Node Features (dim=3):**
- Initial_Population
- Accumulated_Infections  
- Num_Samples

**Node Label:**
- Source_Sink_Score (regression target)

**Edge Features:**
- Phylogenetic distance vectors (variable length, padded to max)

In [ ]:
import argparse
import pandas as pd
import os
from glob import glob
import numpy as np
import dgl
import torch


def parse_edge_feature(edge_feature_str):
    """
    Parse edge feature string from CSV format into float vector.
    
    Args:
        edge_feature_str: String in format [xx\tyy\tzz]
    
    Returns:
        list: Vector of floats
    """
    vector_str = str(edge_feature_str).strip('[]')
    return [float(v) for v in vector_str.split('\t')]


def load_and_convert_to_dgl(input_folder, num_of_nodes, min_length, max_length):
    """
    Load graph data from CSV files, filter, and convert to DGL format.
    
    This function performs the following steps:
    1. Load all node and edge CSV files
    2. Filter for fully connected graphs with specified node count
    3. Filter by edge feature vector length constraints
    4. Pad edge features to uniform length
    5. Convert to DGL bidirectional graphs

    Args:
        input_folder: Path to folder containing *_node.csv and *_edge.csv files
        num_of_nodes: Number of nodes for fully connected graph selection
        min_length: Minimum edge feature vector length
        max_length: Maximum edge feature vector length

    Returns:
        tuple: (dgl_graphs, metadata)
            - dgl_graphs: List of DGL graph objects
            - metadata: Dict with max_length, max_value, num_graphs
    """
    # Load all CSV files
    node_files = glob(os.path.join(input_folder, "*_node.csv"))
    edge_files = glob(os.path.join(input_folder, "*_edge.csv"))

    print(f"Found {len(node_files)} node files and {len(edge_files)} edge files")

    if not node_files or not edge_files:
        print("ERROR: No data files found!")
        return [], {}

    # Concatenate all data
    all_nodes = pd.concat([pd.read_csv(f) for f in node_files], ignore_index=True)
    all_edges = pd.concat([pd.read_csv(f) for f in edge_files], ignore_index=True)

    print(f"Total nodes: {len(all_nodes)}, Total edges: {len(all_edges)}")

    # Filter Step 1: Select fully connected graphs
    # Expected: n self-loops + n*(n-1)/2 undirected edges
    expected_edges = num_of_nodes + num_of_nodes * (num_of_nodes - 1) // 2
    node_counts = all_nodes.groupby('graph_id').size()
    edge_counts = all_edges.groupby('graph_id').size()

    fully_connected_ids = {
        gid for gid in node_counts.index
        if node_counts[gid] == num_of_nodes and edge_counts.get(gid, 0) == expected_edges
    }

    print(f"Fully connected graphs ({num_of_nodes} nodes, {expected_edges} edges): {len(fully_connected_ids)}")

    # Filter Step 2: Edge feature length constraints
    # Parse edge features and compute statistics in a single pass
    valid_graphs = {}
    actual_max_length = 0
    all_edge_values = []

    for graph_id in fully_connected_ids:
        graph_edges = all_edges[all_edges['graph_id'] == graph_id]
        edge_vectors = [parse_edge_feature(ef) for ef in graph_edges['edge_features']]

        # Check length constraints for this graph
        lengths = [len(v) for v in edge_vectors]
        min_len, max_len = min(lengths), max(lengths)

        if min_len >= min_length and max_len <= max_length:
            valid_graphs[graph_id] = edge_vectors
            actual_max_length = max(actual_max_length, max_len)
            all_edge_values.extend([val for vec in edge_vectors for val in vec])

    print(f"Valid graphs after length filter [{min_length}, {max_length}]: {len(valid_graphs)}")

    if not valid_graphs:
        print("WARNING: No graphs passed filtering criteria")
        return [], {}

    # Compute padding value
    max_value = max(all_edge_values)
    print(f"Edge feature padding: MAX_LENGTH={actual_max_length}, MAX_VALUE={max_value:.2f}")

    # Convert to DGL graphs
    dgl_graphs = []

    for graph_id in valid_graphs.keys():
        # Get node and edge data for this graph
        node_df = all_nodes[all_nodes['graph_id'] == graph_id].sort_values('node').reset_index(drop=True)
        edge_df = all_edges[all_edges['graph_id'] == graph_id]

        # Extract node features (3D) and labels
        node_features = node_df[['Initial_Population', 'Accumulated_Infections', 'Num_Samples']].values.astype(np.float32)
        node_labels = node_df['Source_Sink_Score'].values.astype(np.float32)

        # Extract edge list
        src_nodes = edge_df['node1'].values.astype(np.int64)
        dst_nodes = edge_df['node2'].values.astype(np.int64)

        # Pad edge features to uniform length
        padded_edge_features = [
            vec + [max_value] * (actual_max_length - len(vec))
            for vec in valid_graphs[graph_id]
        ]

        # Create DGL graph
        g = dgl.graph((torch.tensor(src_nodes), torch.tensor(dst_nodes)))
        g.ndata['feat'] = torch.tensor(node_features, dtype=torch.float32)
        g.ndata['label'] = torch.tensor(node_labels, dtype=torch.float32)
        g.edata['feat'] = torch.tensor(padded_edge_features, dtype=torch.float32)

        # Convert to bidirectional graph (makes edges flow both directions)
        g = dgl.to_bidirected(g, copy_ndata=True)
        
        # Handle edge features after bidirectionalization
        # DGL may not automatically copy edge features, so we reconstruct them
        if 'feat' not in g.edata or g.edata['feat'].shape[0] != g.num_edges():
            edge_to_feat = {(s, d): g.edata['feat'][i] if 'feat' in g.edata else torch.tensor(padded_edge_features[i]) 
                           for i, (s, d) in enumerate(zip(src_nodes, dst_nodes))}
            
            new_edge_feats = []
            for s, d in zip(*g.edges()):
                s, d = s.item(), d.item()
                # Use original edge feature or its reverse
                feat = edge_to_feat.get((s, d), edge_to_feat.get((d, s)))
                new_edge_feats.append(feat)
            
            g.edata['feat'] = torch.stack(new_edge_feats)
        
        g.graph_id = graph_id
        dgl_graphs.append(g)

    print(f"\nConverted {len(dgl_graphs)} graphs to DGL format")
    print(f"  Node feature dim: 3, Edge feature dim: {actual_max_length}")

    metadata = {
        'max_length': actual_max_length,
        'max_value': max_value,
        'num_graphs': len(dgl_graphs)
    }

    return dgl_graphs, metadata


def main(input_folder, num_of_nodes, min_length, max_length):
    """
    Main entry point for data loading pipeline.
    
    Args:
        input_folder: Path to data folder
        num_of_nodes: Number of nodes per graph
        min_length: Minimum edge feature length
        max_length: Maximum edge feature length
        
    Returns:
        tuple: (dgl_graphs, metadata)
    """
    dgl_graphs, metadata = load_and_convert_to_dgl(input_folder, num_of_nodes, min_length, max_length)

    if dgl_graphs:
        # After bidirectionalization: n self-loops + n*(n-1) directed edges
        edges_after_bidirected = num_of_nodes + num_of_nodes * (num_of_nodes - 1)
        
        print(f"\n{'='*60}")
        print("DATA LOADING SUMMARY")
        print(f"{'='*60}")
        print(f"  Total graphs loaded:     {len(dgl_graphs)}")
        print(f"  Nodes per graph:         {num_of_nodes}")
        print(f"  Edges per graph:         {edges_after_bidirected} (bidirectional)")
        print(f"  Node features (dim=3):   [Initial_Population, Accumulated_Infections, Num_Samples]")
        print(f"  Node label:              Source_Sink_Score")
        print(f"  Edge feature dim:        {metadata['max_length']} (padded with {metadata['max_value']:.2f})")
        print(f"{'='*60}\n")
        
    return dgl_graphs, metadata

## Load the Data

Execute the data loading pipeline with filtering parameters:

**Parameters:**
- `input_folder`: Directory containing `*_node.csv` and `*_edge.csv` files
- `num_of_nodes`: Filter for graphs with exactly this many nodes (e.g., 10)
- `min_length`: Minimum acceptable edge feature vector length
- `max_length`: Maximum acceptable edge feature vector length

**Filtering Process:**
1. Load all CSV files and concatenate
2. Select fully connected graphs (correct node and edge counts)
3. Filter by edge feature length constraints
4. Pad edge features to uniform length
5. Convert to DGL bidirectional graphs

In [2]:
# Run the main function with specified parameters
input_folder = '/Users/lukelyu/Desktop/epidata/multi'
num_of_nodes = 10
min_length = 0
max_length = 5000
dgl_graphs, metadata = main(input_folder, num_of_nodes, min_length, max_length)

Found 10 node files and 10 edge files
Total nodes: 1000, Total edges: 5435
Graphs with 10 nodes and 55 edges: 94
Graphs after length filter (0 <= min_len and max_len <= 5000): 78

Edge feature padding: MAX_LENGTH=4988, MAX_VALUE=559.11
Converted 78 graphs to DGL format
  Node feature dim: 3, Edge feature dim: 4988

Summary:
  Total graphs: 78
  Nodes per graph: 10
  Edges per graph (after bidirectionalization): 100
  Node features: [Initial_Population, Accumulated_Infections, Num_Samples]
  Node label: Source_Sink_Score
  Edge feature dim: 4988 (padded with 559.11)


## Inspect Metadata

The `metadata` dictionary contains information about the loaded dataset:

- `max_length`: Maximum edge feature vector length (after filtering, before padding)
- `max_value`: Maximum value in all edge features (used for padding)
- `num_graphs`: Total number of graphs successfully loaded

In [ ]:
# Examine metadata
print("Metadata keys:", list(metadata.keys()))
print(f"\nMaximum edge feature length: {metadata['max_length']}")
print(f"Maximum value (for padding):  {metadata['max_value']:.2f}")
print(f"Number of graphs loaded:      {metadata['num_graphs']}")

## Examine Graph Structure

Inspect the structure of loaded DGL graphs:
- Number of nodes and edges
- Node data: features (Initial_Population, Accumulated_Infections, Num_Samples) and labels (Source_Sink_Score)
- Edge data: feature vectors
- Sample values from the first graph

In [ ]:
# Examine the first graph
print(f"Total graphs loaded: {len(dgl_graphs)}\n")

g = dgl_graphs[0]
print(f"First graph: {g.graph_id}")
print(f"  Nodes: {g.num_nodes()}")
print(f"  Edges: {g.num_edges()}")

# Node data
print(f"\nNode data:")
print(f"  Features shape: {g.ndata['feat'].shape}  # (10 nodes, 3 features)")
print(f"  Labels shape:   {g.ndata['label'].shape}  # (10 nodes)")
print(f"  First 3 node features:\n{g.ndata['feat'][:3]}")
print(f"  First 3 node labels: {g.ndata['label'][:3]}")

# Edge data
print(f"\nEdge data:")
print(f"  Features shape: {g.edata['feat'].shape}  # (100 edges, {g.edata['feat'].shape[1]} features)")

## GAT Model Architecture

**Overview:**
This section defines a Graph Attention Network (GAT) for node-level regression. The model predicts `Source_Sink_Score` for each node using both node features and edge features (phylogenetic distances).

**Key Components:**

1. **EdgeInformedGATLayer**: Custom GAT layer where edge features directly contribute to attention weights
   - Attention mechanism combines source node, destination node, and edge information
   - Multi-head attention for learning diverse relationship patterns
   - Softmax normalization ensures attention weights sum to 1

2. **GATModel**: End-to-end model for node regression
   - **Encoder**: Projects 3D node features to 32D hidden space
   - **GAT Layer**: Single-layer message passing with 2 attention heads (32 → 64 dims)
   - **Classifier**: Maps aggregated features to scalar predictions (64 → 8 → 1)

**Why Single Layer?**
Since graphs are small (10 nodes) and fully connected, a single GAT layer is sufficient for nodes to gather information from all neighbors. Deeper architectures risk oversmoothing in dense graphs.

**Model Flow:**
```
Node Features (3) 
    → Encoder + ReLU (32)
    → EdgeInformedGAT with 2 heads (64)
    → Classifier (8 → 1)
    → Source_Sink_Score prediction
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl.function as fn


class EdgeInformedGATLayer(nn.Module):
    """
    Graph Attention Network layer where edge features inform attention weights.
    
    This layer computes attention-weighted message passing where:
    - Node features are transformed and used for attention computation
    - Edge features contribute to attention scores
    - Multi-head attention aggregates neighbor information
    
    Attention formula:
        e_ij = LeakyReLU(a_src^T h_i + a_dst^T h_j + a_edge^T e_ij)
        α_ij = softmax_j(e_ij)
        h_i' = Σ_j α_ij h_j
    """
    
    def __init__(self, in_dim, out_dim, edge_dim, num_heads=1):
        """
        Args:
            in_dim: Input node feature dimension
            out_dim: Output node feature dimension per head
            edge_dim: Edge feature dimension
            num_heads: Number of attention heads
        """
        super(EdgeInformedGATLayer, self).__init__()
        self.num_heads = num_heads
        self.out_dim = out_dim
        
        # Transform node features: in_dim -> (out_dim * num_heads)
        self.fc_node = nn.Linear(in_dim, out_dim * num_heads, bias=False)
        
        # Transform edge features to attention scores: edge_dim -> num_heads
        self.fc_edge = nn.Linear(edge_dim, num_heads, bias=False)
        
        # Attention parameters for source and destination nodes
        self.attn_src = nn.Parameter(torch.FloatTensor(size=(1, num_heads, out_dim)))
        self.attn_dst = nn.Parameter(torch.FloatTensor(size=(1, num_heads, out_dim)))
        
        self.reset_parameters()
    
    def reset_parameters(self):
        """Initialize parameters with Xavier/Glorot initialization."""
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_normal_(self.fc_node.weight, gain=gain)
        nn.init.xavier_normal_(self.fc_edge.weight, gain=gain)
        nn.init.xavier_normal_(self.attn_src, gain=gain)
        nn.init.xavier_normal_(self.attn_dst, gain=gain)
    
    def forward(self, g, node_feat, edge_feat):
        """
        Forward pass with edge-informed attention.
        
        Args:
            g: DGL graph
            node_feat: Node features (N, in_dim)
            edge_feat: Edge features (E, edge_dim)
            
        Returns:
            torch.Tensor: Updated node features (N, out_dim * num_heads)
        """
        with g.local_scope():
            # Transform and reshape: (N, in_dim) -> (N, heads, out_dim)
            h = self.fc_node(node_feat).view(-1, self.num_heads, self.out_dim)
            g.ndata['ft'] = h
            
            # Compute attention contributions from source and destination nodes
            g.ndata['a_src'] = (h * self.attn_src).sum(dim=-1, keepdim=True)  # (N, heads, 1)
            g.ndata['a_dst'] = (h * self.attn_dst).sum(dim=-1, keepdim=True)  # (N, heads, 1)
            
            # Compute edge contribution to attention
            edge_attn = self.fc_edge(edge_feat).unsqueeze(-1)  # (E, heads, 1)
            g.edata['a_edge'] = edge_attn
            
            # Combine all attention components: a_src + a_dst + a_edge
            g.apply_edges(fn.u_add_v('a_src', 'a_dst', 'a_node'))
            g.edata['a'] = F.leaky_relu(g.edata['a_node'] + g.edata['a_edge'])
            
            # Normalize attention weights using softmax
            g.edata['a'] = dgl.nn.functional.edge_softmax(g, g.edata['a'])
            
            # Message passing: aggregate neighbor features weighted by attention
            g.update_all(fn.u_mul_e('ft', 'a', 'm'), fn.sum('m', 'ft'))
            
            # Flatten multi-head outputs: (N, heads, out_dim) -> (N, heads * out_dim)
            return g.ndata['ft'].flatten(1)


class GATModel(nn.Module):
    """
    GAT model for node-level regression with edge-informed attention.
    
    Architecture:
        Input: Node features (dim=3)
          ↓
        Encoder: Linear(3 -> 32) + ReLU
          ↓
        GAT Layer: EdgeInformedGAT(32 -> 32, num_heads=2)
          ↓
        Classifier: Linear(64 -> 8) + ReLU + Linear(8 -> 1)
          ↓
        Output: Node-level predictions (Source_Sink_Score)
    
    The model predicts a continuous score for each node based on:
    - Node features (population, infections, samples)
    - Graph structure (phylogenetic relationships)
    - Edge features (phylogenetic distances)
    """
    
    def __init__(self, node_dim, edge_dim, hidden_dim=32, num_heads=2):
        """
        Args:
            node_dim: Input node feature dimension
            edge_dim: Input edge feature dimension
            hidden_dim: Hidden dimension for GAT layer
            num_heads: Number of attention heads
        """
        super(GATModel, self).__init__()
        
        self.encoder = nn.Linear(node_dim, hidden_dim)
        self.gat = EdgeInformedGATLayer(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * num_heads, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
    
    def forward(self, g, node_feat, edge_feat):
        """
        Forward pass for node-level prediction.
        
        Args:
            g: DGL graph
            node_feat: Node features (N, node_dim)
            edge_feat: Edge features (E, edge_dim)
            
        Returns:
            torch.Tensor: Node-level predictions (N,)
        """
        h = F.relu(self.encoder(node_feat))      # (N, hidden_dim)
        h = self.gat(g, h, edge_feat)            # (N, hidden_dim * num_heads)
        out = self.classifier(h)                  # (N, 1)
        return out.squeeze(-1)                    # (N,)


# Model initialization
node_dim = 3
edge_dim = metadata['max_length']
hidden_dim = 32
num_heads = 2

model = GATModel(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=hidden_dim, num_heads=num_heads)

# Print model summary
print("="*60)
print("MODEL INITIALIZATION")
print("="*60)
print(f"  Node feature dim:        {node_dim}")
print(f"  Edge feature dim:        {edge_dim}")
print(f"  Hidden dimension:        {hidden_dim}")
print(f"  Attention heads:         {num_heads}")
print(f"\n  Architecture:")
print(f"    Input (3) -> Encoder (32) -> GAT (32×2 heads=64) -> Classifier (8) -> Output (1)")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Total parameters:        {total_params:,}")
print(f"  Trainable parameters:    {trainable_params:,}")
print("="*60)
print(f"\nModel structure:\n{model}")